# Potentiostat Testing Notebook

This notebook tests the potentiostat functionality with three experiments:
1. **CA (Chronoamperometry)** - Constant potential at 1V for 60s
2. **CP (Chronopotentiometry)** - Constant current at 5mA
3. **CV (Cyclic Voltammetry)** - Sweep from 0.5V to -0.5V

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time
from ps4_ref import Potentiostat, ADC, DAC, Resistors

# Configure matplotlib for better plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Connect to Potentiostat

**Important**: Update the `serial_port` to match your device (e.g., `/dev/ttyACM0` on Linux, `COM3` on Windows)

In [ ]:
# Update this to your serial port
serial_port = '/dev/ttyACM0'  # Linux/Mac
# serial_port = 'COM3'  # Windows

# Create and connect to potentiostat
ps = Potentiostat(serial_port=serial_port, baudrate=115200, device_ID=1)
print("Connecting to potentiostat...")
ps.connect()
print("Connected successfully!")

# Read OCP
ocp = ps.read_ocp()
print(f"Open Circuit Potential: {ocp:.4f} V")

## 2. Chronoamperometry (CA) - Constant Potential at 1V for 60s

Apply a constant potential of 1V and measure the current over time.

In [ ]:
# CA parameters
ca_potential = 1.0  # V
ca_duration = 60    # seconds
ca_sample_rate = 10 # Hz (samples per second)

print(f"Starting CA: {ca_potential}V for {ca_duration}s at {ca_sample_rate}Hz")

# Prepare for measurement
ps.write_switch(True)  # Connect CE-WE
ps.write_auto_gain(True)  # Enable auto-gain for better current measurement

# Set the potential
ps.write_potential(ca_potential)

# Collect data
num_samples = int(ca_duration * ca_sample_rate)
ca_data = np.zeros((num_samples, 3))  # time, voltage, current

delay_ms = int(1000 / ca_sample_rate)
start_time = time.time()

for i in range(num_samples):
    elapsed = time.time() - start_time
    v_i = ps.read_potential_current()
    ca_data[i] = [elapsed, v_i[0], v_i[1]]
    
    # Progress indicator every 10 samples
    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/{num_samples} samples ({elapsed:.1f}s)")
    
    time.sleep(delay_ms / 1000.0)

print("CA completed!")

# Save data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ca_filename = f"CA_{timestamp}.csv"
np.savetxt(ca_filename, ca_data, delimiter=",", 
           fmt="%.6f,%.6f,%.9f",
           header="time(s),voltage(V),current(A)")
print(f"Data saved to {ca_filename}")

In [ ]:
# Plot CA results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Current vs time
ax1.plot(ca_data[:, 0], ca_data[:, 2] * 1e6, 'b-', linewidth=1)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Current (µA)')
ax1.set_title(f'Chronoamperometry at {ca_potential}V')
ax1.grid(True, alpha=0.3)

# Voltage vs time (should be constant)
ax2.plot(ca_data[:, 0], ca_data[:, 1], 'r-', linewidth=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Voltage (V)')
ax2.set_title('Applied Potential vs Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"CA_{timestamp}.png", dpi=300)
plt.show()

print(f"Average current: {np.mean(ca_data[:, 2])*1e6:.3f} µA")
print(f"Current std dev: {np.std(ca_data[:, 2])*1e6:.3f} µA")

## 3. Chronopotentiometry (CP) - Constant Current at 5mA

Apply a constant current of 5mA and measure the voltage response.

In [ ]:
# CP parameters
cp_current = 5.0  # mA
cp_duration = 60  # seconds
cp_sample_rate = 10  # Hz

print(f"Starting CP: {cp_current}mA for {cp_duration}s")

# Start current hold
ps.write_switch(True)
ps.write_current_hold(target_current_mA=cp_current, 
                       initial_step_V=0.0001, 
                       learning_rate=0.05, 
                       force_gain=False)

print("Current hold started, collecting data...")

# Collect data
num_samples = int(cp_duration * cp_sample_rate)
cp_data = np.zeros((num_samples, 3))  # time, voltage, current

delay_ms = int(1000 / cp_sample_rate)
start_time = time.time()

for i in range(num_samples):
    elapsed = time.time() - start_time
    v_i = ps.read_potential_current()
    cp_data[i] = [elapsed, v_i[0], v_i[1]]
    
    # Progress indicator every 10 samples
    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/{num_samples} samples ({elapsed:.1f}s)")
    
    time.sleep(delay_ms / 1000.0)

# Stop current hold
ps.write_current_hold_stop()
print("CP completed!")

# Save data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
cp_filename = f"CP_{timestamp}.csv"
np.savetxt(cp_filename, cp_data, delimiter=",", 
           fmt="%.6f,%.6f,%.9f",
           header="time(s),voltage(V),current(A)")
print(f"Data saved to {cp_filename}")

In [ ]:
# Plot CP results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Voltage vs time
ax1.plot(cp_data[:, 0], cp_data[:, 1], 'r-', linewidth=1)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Voltage (V)')
ax1.set_title(f'Chronopotentiometry at {cp_current}mA')
ax1.grid(True, alpha=0.3)

# Current vs time (should be constant)
ax2.plot(cp_data[:, 0], cp_data[:, 2] * 1e3, 'b-', linewidth=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Current (mA)')
ax2.set_title('Applied Current vs Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"CP_{timestamp}.png", dpi=300)
plt.show()

print(f"Average voltage: {np.mean(cp_data[:, 1]):.4f} V")
print(f"Voltage std dev: {np.std(cp_data[:, 1]):.4f} V")
print(f"Average current: {np.mean(cp_data[:, 2])*1e3:.3f} mA")

## 4. Cyclic Voltammetry (CV) - 0.5V to -0.5V

Perform cyclic voltammetry sweeping from 0.5V to -0.5V and back.

In [ ]:
# CV parameters
cv_min_V = -0.5  # V
cv_max_V = 0.5   # V
cv_cycles = 3    # Number of cycles
cv_scan_rate = 100  # mV/s
cv_step_hz = 250    # Sampling frequency

print(f"Starting CV: {cv_min_V}V to {cv_max_V}V")
print(f"Scan rate: {cv_scan_rate} mV/s, Cycles: {cv_cycles}")

# Perform CV
cv_data = ps.perform_CV(min_V=cv_min_V, 
                        max_V=cv_max_V, 
                        cycles=cv_cycles, 
                        mV_s=cv_scan_rate, 
                        step_hz=cv_step_hz,
                        start_V=None,  # Start from OCP
                        last_V=None)   # End at OCP

print(f"CV completed! Collected {cv_data.shape[0]} data points")

# Save data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
cv_filename = f"CV_{timestamp}.csv"
np.savetxt(cv_filename, cv_data, delimiter=",", 
           fmt="%.6f,%.6f,%.9f,%d,%d,%.6f",
           header="time(s),voltage(V),current(A),cycle_num,exp_num,voltage_applied(V)")
print(f"Data saved to {cv_filename}")

In [ ]:
# Plot CV results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# CV plot (Current vs Voltage)
colors = plt.cm.viridis(np.linspace(0, 1, cv_cycles))
for cycle in range(cv_cycles):
    cycle_data = cv_data[cv_data[:, 3] == cycle]
    ax1.plot(cycle_data[:, 1], cycle_data[:, 2] * 1e6, 
             color=colors[cycle], linewidth=1.5, label=f'Cycle {cycle+1}')

ax1.set_xlabel('Potential (V vs RE)')
ax1.set_ylabel('Current (µA)')
ax1.set_title(f'Cyclic Voltammetry ({cv_scan_rate} mV/s)')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
ax1.axvline(x=0, color='k', linestyle='--', linewidth=0.5)

# Current vs Time
ax2.plot(cv_data[:, 0], cv_data[:, 2] * 1e6, 'b-', linewidth=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Current (µA)')
ax2.set_title('Current vs Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"CV_{timestamp}.png", dpi=300)
plt.show()

print(f"Peak anodic current: {np.max(cv_data[:, 2])*1e6:.3f} µA")
print(f"Peak cathodic current: {np.min(cv_data[:, 2])*1e6:.3f} µA")

## 5. Disconnect Potentiostat

Always disconnect properly when finished.

In [ ]:
# Turn off switch and disconnect
ps.write_switch(False)
ps.disconnect()
print("Potentiostat disconnected successfully!")

## Summary

This notebook tested three key electrochemical techniques:

1. **CA (Chronoamperometry)** - Measured current response at constant 1V potential
2. **CP (Chronopotentiometry)** - Measured voltage response at constant 5mA current
3. **CV (Cyclic Voltammetry)** - Swept potential from 0.5V to -0.5V

All data has been saved to CSV files with timestamps, and plots have been generated.